# EEG Analysis — OpenNeuro ds001787 (Meditazione)
**Libreria principale:** MNE-Python  
**Pipeline:** Caricamento BIDS → Preprocessing → Analisi PSD

> 💡 Ogni cella ha commenti dettagliati. Eseguile in ordine dall'alto verso il basso.

---
## 0. Installazione dipendenze
Esegui questa cella solo la prima volta.

In [ ]:
# Installa le librerie necessarie (solo la prima volta)
# !pip install mne mne-bids openneuro-py matplotlib numpy scipy

---
## 1. Import librerie

In [ ]:
import mne
import mne_bids
import openneuro
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Silenzia i messaggi di log meno importanti
mne.set_log_level('WARNING')

print(f"MNE version: {mne.__version__}")
print(f"MNE-BIDS version: {mne_bids.__version__}")

---
## 2. Download dataset da OpenNeuro

Il dataset ds001787 è uno studio EEG sulla **meditazione** in formato BIDS.  
La struttura delle cartelle sarà:
```
ds001787/
├── participants.tsv       ← info sui soggetti
├── dataset_description.json
└── sub-01/
    └── eeg/
        ├── sub-01_task-meditation_eeg.set   ← dati EEG
        ├── sub-01_task-meditation_channels.tsv
        └── sub-01_task-meditation_events.tsv
```

In [ ]:
# Cartella dove salvare i dati
bids_root = Path('./ds001787')
bids_root.mkdir(exist_ok=True)

# Scarichiamo solo il primo soggetto per cominciare
# (rimuovi include=[...] per scaricare tutto il dataset)
openneuro.download(
    dataset='ds001787',
    target_dir=bids_root,
    include=['sub-01']  # solo soggetto 1
)

print("Download completato!")

---
## 3. Esplora la struttura del dataset

In [ ]:
# Visualizza la struttura delle cartelle (max 3 livelli)
mne_bids.print_dir_tree(bids_root, max_depth=3)

In [ ]:
import pandas as pd

# Leggi le informazioni sui partecipanti
participants_file = bids_root / 'participants.tsv'
if participants_file.exists():
    df = pd.read_csv(participants_file, sep='\t')
    print(f"Numero soggetti: {len(df)}")
    display(df.head())
else:
    print("File participants.tsv non trovato")

---
## 4. Carica i dati EEG con MNE-BIDS

In [ ]:
# Crea un BIDSPath — è l'oggetto che MNE usa per trovare i file
# Modifica subject e task_name in base al dataset reale
subject = '01'
task_name = 'meditation'  # <-- potrebbe essere diverso, controlla con print_dir_tree

bids_path = mne_bids.BIDSPath(
    subject=subject,
    task=task_name,
    root=bids_root,
    datatype='eeg'
)

print(f"Percorso file: {bids_path.fpath}")

# Carica i dati raw
raw = mne_bids.read_raw_bids(bids_path, verbose=False)

print("\n--- INFO REGISTRAZIONE ---")
print(f"Frequenza di campionamento: {raw.info['sfreq']} Hz")
print(f"Numero canali EEG: {len(raw.ch_names)}")
print(f"Durata: {raw.times[-1]:.1f} secondi ({raw.times[-1]/60:.1f} minuti)")
print(f"Canali: {raw.ch_names}")

---
## 5. Preprocessing

Il preprocessing rimuove il "rumore" dal segnale EEG prima dell'analisi.  
I passaggi standard sono:
1. **Filtro passa-banda** → mantieni solo le frequenze di interesse (1–40 Hz)
2. **Riferimento medio** → ridistribuisce il segnale su tutti gli elettrodi
3. **Identificazione canali cattivi** → elettrodi rumorosi da escludere

In [ ]:
# Copia i dati per non modificare l'originale
raw_preprocessed = raw.copy()

# --- PASSO 1: Carica i dati in memoria ---
raw_preprocessed.load_data()

# --- PASSO 2: Filtro passa-banda 1–40 Hz ---
# - 1 Hz: rimuove deriva lenta (movimento soggetto, sudore)
# - 40 Hz: rimuove rumore ad alta frequenza (muscoli, rete elettrica)
raw_preprocessed.filter(
    l_freq=1.0,    # frequenza minima
    h_freq=40.0,   # frequenza massima
    method='fir',
    verbose=False
)
print("✅ Filtro 1-40 Hz applicato")

# --- PASSO 3: Notch filter per rimuovere la frequenza di rete (50 Hz in Europa) ---
raw_preprocessed.notch_filter(freqs=50.0, verbose=False)
print("✅ Notch filter 50 Hz applicato")

# --- PASSO 4: Ri-referenziazione alla media ---
# Il segnale EEG è relativo a un riferimento: usiamo la media di tutti gli elettrodi
raw_preprocessed.set_eeg_reference('average', projection=False, verbose=False)
print("✅ Riferimento medio applicato")

print("\n🎉 Preprocessing completato!")

In [ ]:
# Visualizza il segnale grezzo vs preprocessato per confronto
# (mostra i primi 10 secondi, primi 5 canali)
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

n_channels_to_plot = min(5, len(raw.ch_names))
t_start, t_end = 0, 10  # secondi
sfreq = raw.info['sfreq']

start_idx = int(t_start * sfreq)
end_idx = int(t_end * sfreq)
times = raw.times[start_idx:end_idx]

# Segnale grezzo
data_raw, _ = raw[:n_channels_to_plot, start_idx:end_idx]
for i, ch in enumerate(raw.ch_names[:n_channels_to_plot]):
    axes[0].plot(times, data_raw[i] * 1e6 + i * 100, label=ch, lw=0.8)
axes[0].set_title('Segnale GREZZO', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Ampiezza (µV)')
axes[0].legend(loc='upper right', fontsize=7)

# Segnale preprocessato
data_prep, _ = raw_preprocessed[:n_channels_to_plot, start_idx:end_idx]
for i, ch in enumerate(raw_preprocessed.ch_names[:n_channels_to_plot]):
    axes[1].plot(times, data_prep[i] * 1e6 + i * 100, label=ch, lw=0.8)
axes[1].set_title('Segnale PREPROCESSATO (filtro 1–40 Hz)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Ampiezza (µV)')
axes[1].set_xlabel('Tempo (s)')
axes[1].legend(loc='upper right', fontsize=7)

plt.tight_layout()
plt.savefig('confronto_raw_preprocessed.png', dpi=150, bbox_inches='tight')
plt.show()
print("Grafico salvato come 'confronto_raw_preprocessed.png'")

---
## 6. Rimozione artefatti con Z-score per epoche da 2 secondi

### Cos'è un artefatto EEG?
Un **artefatto** è un segnale spurio che non proviene dall'attività cerebrale:  
movimenti degli occhi (EOG), contrazioni muscolari (EMG), cavi che si muovono, battito cardiaco, ecc.

### Strategia: Z-score per epoca
1. Dividiamo il segnale continuo in **epoche da 2 secondi**
2. Per ogni epoca e ogni canale, calcoliamo il **picco di ampiezza** (max - min)
3. Calcoliamo lo **z-score** di quel picco *rispetto a tutte le epoche dello stesso canale*
4. Se lo z-score supera una **soglia** (default: 3.0), l'epoca è considerata artefatta

```
z = (picco_epoca - media_picchi) / std_picchi
```
> Un z-score > 3 significa che il picco è oltre 3 deviazioni standard dalla media → molto probabilmente un artefatto.

In [ ]:
# ─────────────────────────────────────────────
# PARAMETRI — modificabili
# ─────────────────────────────────────────────
EPOCH_DURATION = 2.0   # durata di ogni epoca in secondi
Z_THRESHOLD    = 3.0   # soglia z-score (3.0 = conservativo, 2.5 = più aggressivo)
# ─────────────────────────────────────────────

sfreq = raw_preprocessed.info['sfreq']
epoch_samples = int(EPOCH_DURATION * sfreq)  # campioni per epoca

# Ottieni i dati EEG come array numpy: shape (n_canali, n_campioni)
data, times_raw = raw_preprocessed[:, :]
n_channels, n_samples = data.shape

# Calcola quante epoche intere possiamo ricavare
n_epochs = n_samples // epoch_samples
print(f"Campioni totali: {n_samples}")
print(f"Campioni per epoca ({EPOCH_DURATION}s): {epoch_samples}")
print(f"Numero epoche analizzabili: {n_epochs}")

In [ ]:
# ── PASSO 1: calcola il picco (peak-to-peak) per ogni epoca e canale ──
# peak-to-peak = max(segnale) - min(segnale) nell'epoca
# shape risultante: (n_canali, n_epoche)

peak_to_peak = np.zeros((n_channels, n_epochs))

for ep in range(n_epochs):
    start = ep * epoch_samples
    end   = start + epoch_samples
    epoch_data = data[:, start:end]                         # shape: (n_canali, epoch_samples)
    peak_to_peak[:, ep] = epoch_data.max(axis=1) - epoch_data.min(axis=1)

print(f"Peak-to-peak calcolato → shape: {peak_to_peak.shape}  (canali × epoche)")

In [ ]:
# ── PASSO 2: calcola lo z-score per canale ──
# Per ogni canale: z = (valore - media) / std
# Lo z-score è calcolato LUNGO le epoche (axis=1), non tra canali

mean_ptp = peak_to_peak.mean(axis=1, keepdims=True)   # shape: (n_canali, 1)
std_ptp  = peak_to_peak.std(axis=1, keepdims=True)    # shape: (n_canali, 1)

# Evita divisione per zero se un canale è piatto
std_ptp[std_ptp == 0] = 1e-10

z_scores = (peak_to_peak - mean_ptp) / std_ptp        # shape: (n_canali, n_epoche)

print(f"Z-scores calcolati → shape: {z_scores.shape}")
print(f"Z-score max globale: {z_scores.max():.2f}")
print(f"Z-score min globale: {z_scores.min():.2f}")

In [ ]:
# ── PASSO 3: identifica le epoche da rigettare ──
# Un'epoca viene rigettata se ALMENO UN canale supera la soglia

# Maschera booleana: True = artefatto in quel canale/epoca
artifact_mask = z_scores > Z_THRESHOLD          # shape: (n_canali, n_epoche)

# Epoche con almeno 1 canale artefatto
epochs_to_reject = np.any(artifact_mask, axis=0)  # shape: (n_epoche,)
epochs_clean     = ~epochs_to_reject

n_rejected = epochs_to_reject.sum()
n_clean    = epochs_clean.sum()
pct_rejected = 100 * n_rejected / n_epochs

print(f"\n{'='*45}")
print(f"Soglia z-score:          {Z_THRESHOLD}")
print(f"Epoche totali:           {n_epochs}")
print(f"Epoche RIGETTATE:        {n_rejected}  ({pct_rejected:.1f}%)")
print(f"Epoche PULITE:           {n_clean}  ({100-pct_rejected:.1f}%)")
print(f"{'='*45}")

if pct_rejected > 40:
    print("⚠️  Più del 40% rigettato — considera di alzare la soglia (es. 3.5 o 4.0)")
elif pct_rejected < 5:
    print("ℹ️  Meno del 5% rigettato — soglia conservativa, va bene")
else:
    print("✅  Percentuale di rigetto nella norma")

In [ ]:
# ── PASSO 4: visualizza quali epoche sono state rigettate ──
fig, axes = plt.subplots(2, 1, figsize=(14, 7))

# --- Subplot 1: heatmap z-score (canali × epoche) ---
epoch_indices = np.arange(n_epochs)
im = axes[0].imshow(
    z_scores,
    aspect='auto',
    cmap='RdYlGn_r',       # rosso = alto z-score, verde = basso
    vmin=0, vmax=Z_THRESHOLD * 1.5,
    interpolation='nearest'
)
plt.colorbar(im, ax=axes[0], label='Z-score')
axes[0].axhline(y=-0.5, color='k', lw=0.5)
axes[0].set_xlabel('Epoca (2s)')
axes[0].set_ylabel('Canale')
axes[0].set_title(f'Heatmap Z-score per canale × epoca  |  soglia = {Z_THRESHOLD}', fontsize=11)

# Linea orizzontale per la soglia (proiettata sulle epoche con max z)
max_z_per_epoch = z_scores.max(axis=0)

# --- Subplot 2: max z-score per epoca con soglia ----
axes[1].bar(epoch_indices[epochs_clean],
            max_z_per_epoch[epochs_clean],
            color='#2ecc71', alpha=0.7, width=1.0, label='Epoca pulita')
axes[1].bar(epoch_indices[epochs_to_reject],
            max_z_per_epoch[epochs_to_reject],
            color='#e74c3c', alpha=0.85, width=1.0, label='Epoca rigettata')
axes[1].axhline(y=Z_THRESHOLD, color='black', lw=1.5,
                linestyle='--', label=f'Soglia z={Z_THRESHOLD}')
axes[1].set_xlabel('Epoca (2s)', fontsize=11)
axes[1].set_ylabel('Z-score massimo', fontsize=11)
axes[1].set_title(f'Max z-score per epoca  |  {n_rejected}/{n_epochs} rigettate ({pct_rejected:.1f}%)', fontsize=11)
axes[1].legend(fontsize=9)
axes[1].grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('artifact_rejection_zscore.png', dpi=150, bbox_inches='tight')
plt.show()
print("Grafico salvato come 'artifact_rejection_zscore.png'")

In [ ]:
# ── PASSO 5: ricostruisci il segnale pulito ──
# Concateniamo solo le epoche pulite → segnale continuo senza artefatti

clean_segments = []
for ep in range(n_epochs):
    if epochs_clean[ep]:
        start = ep * epoch_samples
        end   = start + epoch_samples
        clean_segments.append(data[:, start:end])

# Unisci tutti i segmenti puliti
data_clean = np.concatenate(clean_segments, axis=1)  # shape: (n_canali, n_campioni_puliti)

duration_orig  = n_samples / sfreq
duration_clean = data_clean.shape[1] / sfreq

print(f"Durata originale:  {duration_orig:.1f}s  ({duration_orig/60:.1f} min)")
print(f"Durata dopo pulizia: {duration_clean:.1f}s  ({duration_clean/60:.1f} min)")
print(f"Segnale mantenuto: {100*duration_clean/duration_orig:.1f}%")

# Crea un nuovo oggetto RawArray MNE con i dati puliti
raw_clean = mne.io.RawArray(data_clean, raw_preprocessed.info, verbose=False)
print("\n✅ Oggetto raw_clean creato — pronto per la PSD!")

In [ ]:
# ── RIEPILOGO PER CANALE: quante epoche artefatte per ciascuno? ──
import pandas as pd

artifacts_per_channel = artifact_mask.sum(axis=1)  # n. epoche artefatte per canale
pct_per_channel = 100 * artifacts_per_channel / n_epochs

df_artifacts = pd.DataFrame({
    'Canale': raw_preprocessed.ch_names,
    'Epoche artefatte': artifacts_per_channel,
    '% sul totale': pct_per_channel.round(1)
}).sort_values('Epoche artefatte', ascending=False)

print("\n--- TOP 10 canali più rumorosi ---")
display(df_artifacts.head(10))

# Segnala canali molto rumorosi (>30% epoche artefatte)
bad_channels = df_artifacts[df_artifacts['% sul totale'] > 30]['Canale'].tolist()
if bad_channels:
    print(f"\n⚠️  Canali con >30% epoche artefatte (considera di escluderli):")
    print(f"   {bad_channels}")
else:
    print("\n✅ Nessun canale eccessivamente rumoroso (>30%)")

---
## 7. Analisi delle frequenze — PSD (Power Spectral Density)

La PSD misura **quanta energia** è presente a ciascuna frequenza nel segnale EEG.  

> 🧹 Da questo punto usiamo **`raw_clean`** — il segnale già ripulito dagli artefatti.

Le **bande di frequenza EEG** principali:

| Banda | Range (Hz) | Associata a |
|-------|-----------|-------------|
| Delta | 0.5 – 4 | Sonno profondo |
| **Theta** | **4 – 8** | **Meditazione, creatività** |
| **Alpha** | **8 – 13** | **Rilassamento, meditazione** |
| Beta | 13 – 30 | Attenzione, pensiero attivo |
| Gamma | 30 – 40 | Processi cognitivi superiori |

> In questo dataset di meditazione, ci aspettiamo un **picco alpha (~10 Hz)** prominente.

In [ ]:
# Calcola la PSD con il metodo Welch
# (divide il segnale in segmenti sovrapposti e ne media gli spettri)
psd = raw_clean.compute_psd(
    method='welch',
    fmin=1.0,    # frequenza minima da calcolare
    fmax=40.0,   # frequenza massima da calcolare
    n_fft=2048,  # risoluzione frequenziale (più alto = più dettaglio)
    n_overlap=512
)

print("PSD calcolata!")
print(f"Shape dati PSD: {psd.get_data().shape}  (canali × frequenze)")
print(f"Risoluzione frequenziale: {psd.freqs[1] - psd.freqs[0]:.3f} Hz")

In [ ]:
# Visualizzazione PSD — media su tutti i canali
fig, ax = plt.subplots(figsize=(12, 5))

# Ottieni dati e frequenze
psd_data = psd.get_data()  # shape: (n_canali, n_freq)
freqs = psd.freqs

# Converti in dB: 10 * log10(power)
psd_db = 10 * np.log10(psd_data)

# Media e deviazione standard tra canali
psd_mean = psd_db.mean(axis=0)
psd_std = psd_db.std(axis=0)

# Plot linea media
ax.plot(freqs, psd_mean, color='#2f4a8c', lw=2, label='Media tutti i canali')

# Banda di incertezza (±1 std)
ax.fill_between(freqs, psd_mean - psd_std, psd_mean + psd_std,
                alpha=0.2, color='#2f4a8c', label='±1 SD')

# Colora le bande di frequenza
bands = {
    'Delta\n(0.5–4)': (0.5, 4, '#e8d5f5'),
    'Theta\n(4–8)': (4, 8, '#d5e8f5'),
    'Alpha\n(8–13)': (8, 13, '#d5f5e3'),
    'Beta\n(13–30)': (13, 30, '#f5f0d5'),
    'Gamma\n(30–40)': (30, 40, '#f5d5d5'),
}

for band_name, (f_lo, f_hi, color) in bands.items():
    ax.axvspan(f_lo, f_hi, alpha=0.3, color=color)
    ax.text((f_lo + f_hi) / 2, ax.get_ylim()[0] if ax.get_ylim()[0] > -100 else psd_mean.min() - 2,
            band_name, ha='center', va='bottom', fontsize=8, color='#555')

ax.set_xlabel('Frequenza (Hz)', fontsize=12)
ax.set_ylabel('Potenza (dB)', fontsize=12)
ax.set_title('Power Spectral Density — Media su tutti i canali\nds001787 (Meditazione)', fontsize=13)
ax.legend(fontsize=10)
ax.set_xlim(1, 40)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('psd_globale.png', dpi=150, bbox_inches='tight')
plt.show()
print("Grafico salvato come 'psd_globale.png'")

In [ ]:
# Calcola la potenza relativa per ciascuna banda
# (utile per confronti tra soggetti e condizioni)

def band_power(psd_data, freqs, fmin, fmax):
    """Calcola potenza media nella banda [fmin, fmax]."""
    idx = np.logical_and(freqs >= fmin, freqs <= fmax)
    return psd_data[:, idx].mean(axis=1)  # media sulle frequenze, per canale

band_definitions = {
    'Delta (0.5–4 Hz)':   (0.5, 4),
    'Theta (4–8 Hz)':     (4, 8),
    'Alpha (8–13 Hz)':    (8, 13),
    'Beta (13–30 Hz)':    (13, 30),
    'Gamma (30–40 Hz)':   (30, 40),
}

print("--- POTENZA PER BANDA (media su tutti i canali) ---\n")
results = {}
for band_name, (fmin, fmax) in band_definitions.items():
    power = band_power(psd_data, freqs, fmin, fmax)
    mean_power_db = 10 * np.log10(power.mean())
    results[band_name] = mean_power_db
    print(f"{band_name:25s}: {mean_power_db:.2f} dB")

In [ ]:
# Barplot delle bande di frequenza
fig, ax = plt.subplots(figsize=(8, 5))

colors = ['#9b59b6', '#3498db', '#2ecc71', '#f39c12', '#e74c3c']
bands_short = ['Delta', 'Theta', 'Alpha', 'Beta', 'Gamma']
values = list(results.values())

bars = ax.bar(bands_short, values, color=colors, edgecolor='white', linewidth=1.5)

# Aggiungi valori sopra le barre
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{val:.1f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_ylabel('Potenza media (dB)', fontsize=12)
ax.set_title('Confronto potenza per banda di frequenza\nds001787 — sub-01', fontsize=12)
ax.grid(True, axis='y', alpha=0.3)
ax.set_ylim(min(values) - 5, max(values) + 5)

plt.tight_layout()
plt.savefig('band_power_barplot.png', dpi=150, bbox_inches='tight')
plt.show()
print("Grafico salvato come 'band_power_barplot.png'")

---
## 8. Topografia della potenza Alpha

La **topografia** mostra come la potenza è distribuita sullo scalpo.  
In meditazione, l'alpha è tipicamente più forte nelle regioni posteriori (occipitali).

In [ ]:
# Visualizza le topografie per ogni banda
# (richiede che le posizioni degli elettrodi siano definite nel dataset)
try:
    fig = psd.plot_topomap(
        bands={
            'Theta (4-8 Hz)': (4, 8),
            'Alpha (8-13 Hz)': (8, 13),
            'Beta (13-30 Hz)': (13, 30),
        },
        normalize=True,  # normalizza rispetto alla potenza totale
        show=True
    )
    fig.savefig('topografie_bande.png', dpi=150, bbox_inches='tight')
    print("Topografie salvate come 'topografie_bande.png'")
except Exception as e:
    print(f"⚠️ Topografia non disponibile: {e}")
    print("Le posizioni degli elettrodi potrebbero non essere definite nel dataset.")

---
## 9. Riepilogo e prossimi passi

✅ **Cosa abbiamo fatto:**
- Scaricato e caricato il dataset BIDS
- Preprocessing: filtro 1-40 Hz, notch 50 Hz, riferimento medio
- **Artifact rejection con z-score** su epoche da 2s (per canale)
- Calcolato la PSD sul segnale pulito con il metodo Welch
- Visualizzato la distribuzione di potenza per banda

🔜 **Prossimi passi possibili:**
- **ICA**: rimozione artefatti oculari/cardiaci con Independent Component Analysis
- **Epoching**: se il dataset ha eventi, dividere il segnale in finestre temporali
- **Confronto gruppi**: meditatori esperti vs novizi (se il dataset ha questa info)
- **Analisi temporale**: come cambia la potenza alpha durante la sessione?
- **Coerenza**: quanto sono sincronizzati diversi canali cerebrali?